In [ ]:
import logging

FORMAT = '%(asctime)s %(name)s %(funcName)s %(message)s'
log_level = logging.WARNING
logging.basicConfig(format=FORMAT, datefmt='%H:%M:%S',
                    level=log_level)

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import h5py
import numpy as np
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import csv
from pathlib import Path 
from scipy.spatial import distance

bnsi_path = '/scicore/home/nimwegen/degroo0000/Bonsai-data-representation'
sys.path.append(bnsi_path)
from bonsai_scout.bonsai_scout_helpers import Bonvis_figure, Bonvis_settings, Bonvis_metadata

In [ ]:
from paper_figure_scripts_and_notebooks.simulating_datasets.analyzing_simulated_datasets.knn_recall_helpers import get_pdists_on_tree

#### Set some parameters

In [ ]:
SINGLE_DATASET = False
SAVE_FIG = False

## Create Bonsai visualization of ground truth dataset

In [ ]:
if not SINGLE_DATASET:
    dataset_ids_gt = ['simulated_datasets/simulated_binary_10_gens_samplingnoise_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_unbalanced_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_realcovariance_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_seed_2462',
    'simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_realcovariance_seed_2462']

    dataset_descr_lst_gt = ['Perfect binary', 'Unbalanced (Unb)', 'Random branch lengths (Rbl)', 'Real covariance (Reco)', 'Rbl + Unb', 'Rbl + Unb + Reco']
else:
    dataset_ids_gt = ['simulated_datasets/simulated_binary_10_gens_samplingnoise_randomtimes_unbalanced_realcovariance_seed_2462']
    dataset_ids_gt = ['simulated_datasets/simulated_binary_10_gens_samplingnoise_seed_2462']
    # dataset_ids_gt = []
    
    dataset_descr_lst_gt = ['Rbl + Unb + Reco']
    dataset_descr_lst_gt = ['Perfect binary']
    # dataset_descr_lst_gt = []

input_data_folders_gt = [os.path.join('/scicore/home/nimwegen/GROUP/Projects/bonsai_runs/paper_figures_datasets', id, 'UMI_counts') for id in dataset_ids_gt]
bonsai_results_folders_gt = [os.path.join('/scicore/home/nimwegen/GROUP/Projects/bonsai_runs/paper_figures_datasets', id, 'UMI_counts', 'true_tree') for id in dataset_ids_gt]



## Create pairwise distance plots

In [ ]:
cell_ids_lst = []
for ind_dataset, input_data_folder in enumerate(input_data_folders_gt):    
    print(ind_dataset)
    # Load data; extract UMI-counts and cell-IDs
    umi_counts_df = pd.read_csv(os.path.join(input_data_folder, 'Gene_table.txt'), header=0,
                                        index_col=0, sep='\t')
    cell_ids = list(umi_counts_df.columns)
    # umi_counts = umi_counts_df.values
    del umi_counts_df
    
    cell_ids_lst.append(cell_ids)
    # umi_counts_lst.append(umi_counts)

In [ ]:
# Read in ground truth squared pairwise distances (divided by the number of dimensions)
true_dists_lst = []
# selected_gene_inds_lst = []
for ind_dataset, input_data_folder in enumerate(input_data_folders_gt):
    print(ind_dataset)
    delta_gc_true = pd.read_csv(os.path.join(input_data_folder, 'delta_true.txt'), header=None,
                                index_col=None, sep='\t').values
    
    num_dims = delta_gc_true.shape[0]
    true_dists = distance.pdist(delta_gc_true.T, metric='sqeuclidean')/num_dims
    true_dists_lst.append(true_dists)

In [ ]:
# Calculate distances on ground truth tree
from scipy.spatial.distance import squareform

true_tree_dists_lst = []
for ind_dataset, bonsai_results_folder in enumerate(bonsai_results_folders_gt):
    print(ind_dataset)
    cell_ids = cell_ids_lst[ind_dataset]
    tree_results = os.path.join(bonsai_results_folder, 'final_bonsai')
    true_tree_dists = get_pdists_on_tree(os.path.join(tree_results, 'tree.nwk'), cell_ids)
    true_tree_dists_lst.append(true_tree_dists)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Build a 40-color palette from tab20 + tab20b
cmap1 = plt.get_cmap("tab10")
cmap2 = plt.get_cmap("tab20b")

colors = np.vstack([
    cmap1(np.linspace(0, 1, 10)),
    cmap2(np.linspace(0, 1, 20))
])

# Pick color by index
def get_color(ind):
    return colors[ind % 40]

In [ ]:
ncols=2
nrows=3
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 8 * nrows), squeeze=False)
axs = axs.flatten()
for ind_dataset, true_dists in enumerate(true_dists_lst):
    ax = axs[ind_dataset]
    sq_dists = np.asarray(true_tree_dists_lst[ind_dataset])     # visualized distances (x)
    sq_true_dists = np.asarray(true_dists)       # true distances (y)

    # ---- Scatter ----
    ax.scatter(sq_dists, sq_true_dists, s=2, alpha=.01)
    tick_labels = ax.get_xticks()
    # ---- Define bins along x-axis ----
    n_bins = 10
    bins = np.linspace(sq_dists.min() - 1, sq_dists.max() + 1, n_bins + 1)
    # bins = np.quantile(sq_dists, np.linspace(0, 1, n_bins + 1))


    bin_indices = np.digitize(sq_dists, bins) - 1
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    # Collect data per x-bin
    data = [
        sq_true_dists[bin_indices == i]
        for i in range(n_bins)
    ]
    
    # Remove empty bins
    valid = [len(d) > 0 for d in data]
    data = [d for d, v in zip(data, valid) if v]
    positions = [c for c, v in zip(bin_centers, valid) if v]
    
    # ---- Draw vertical boxplots ----
    bp = ax.boxplot(
        data,
        positions=positions,
        widths=(bins[1] - bins[0]) * 0.8,
        vert=True,
        patch_artist=True,
        showfliers=False
    )
    
    # ---- Styling ----
    for box in bp['boxes']:
        box.set_facecolor('tab:orange')
        box.set_alpha(0.6)
        box.set_edgecolor('black')
    
    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(1.3)
    
    ax.set_xlabel('Summed branch lengths (true tree)', fontsize=14)
    ax.set_ylabel('Squared Euclidean distances (ground truth)', fontsize=14)
    ax.set_title(dataset_descr_lst_gt[ind_dataset], fontsize=16)
    ax.set_xticks(tick_labels, tick_labels)
    ax.set_xlim(0)

# plt.show()

In [ ]:
# Store figure.
fig.savefig('overview_figures/euclidean_dist_branch_lengths_ground_truth.png', dpi=300)
fig.savefig('overview_figures/euclidean_dist_branch_lengths_ground_truth.svg')